# Fake Jobs – Cleaned
- Numerische/enkodierte Features, Freitexte entfernt
- Split 50/20/30 pro `SEED`; Imputation, Frequency-Encoding und Scaler werden **nur auf Train** gefittet

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = int(os.environ.get("SEED", 1))
print("SEED", SEED)

## Rohdaten laden
- Label `fraudulent`, stabile `row_id` aus `job_id`

In [ ]:
df = pd.read_csv("../../data/raw/fake_job_postings.csv")
df["row_id"] = df["job_id"].astype("int64")
y = df["fraudulent"]
print(df.shape, "| Outlier-Rate:", round(y.mean(), 4))

## Feature Engineering
- `location` → country/state/city; `salary_range` → `salary_avg` + `salary_missing`
- Rein zeilenweise, damit seed-unabhängig

In [ ]:
parts = df["location"].fillna("").str.split(",", n=2, expand=True).reindex(columns=[0, 1, 2])
df["country"] = parts[0].str.strip().replace("", "missing")
df["state"] = parts[1].fillna("").str.strip().replace("", "missing")
df["city"] = parts[2].fillna("").str.strip().replace("", "missing")

sal = df["salary_range"].fillna("").str.split("-", n=1, expand=True).reindex(columns=[0, 1])
df["salary_avg"] = pd.concat([pd.to_numeric(sal[0], errors="coerce"),
                              pd.to_numeric(sal[1], errors="coerce")], axis=1).mean(axis=1)
df["salary_missing"] = df["salary_avg"].isna().astype(int)

## Spalten droppen & Kategorien vorbereiten
- IDs, Roh-Strukturspalten und alle 5 Freitexte entfernen

In [ ]:
df = df.drop(columns=["job_id", "location", "salary_range", "fraudulent",
                      "title", "company_profile", "description", "requirements", "benefits"])

cat_cols = ["industry", "function", "department", "country", "state", "city",
            "employment_type", "required_experience", "required_education"]
binary = ["telecommuting", "has_company_logo", "has_questions", "salary_missing"]
for c in cat_cols:
    df[c] = df[c].fillna("missing").replace("", "missing")

## Split 50/20/30
- Stratifiziert über das Label, Schlüssel `row_id`
- Zeilen-Universum (seed-unabhängig) und Split-Zuordnung (pro Seed) werden separat abgelegt

In [ ]:
rows = pd.DataFrame({"row_id": df["row_id"].values, "fraudulent": y.values})
tr_id, rest_id = train_test_split(rows["row_id"], train_size=0.5,
                                  stratify=rows["fraudulent"], random_state=SEED)
rest = rows[rows["row_id"].isin(rest_id)]
val_id, te_id = train_test_split(rest["row_id"], train_size=0.4,
                                 stratify=rest["fraudulent"], random_state=SEED)

split = rows[["row_id"]].copy()
split["split"] = np.where(split["row_id"].isin(tr_id), "train",
                          np.where(split["row_id"].isin(val_id), "val", "test"))
tr = df["row_id"].isin(tr_id)

os.makedirs("../../data/splits", exist_ok=True)
rows.to_csv("../../data/splits/rows_fake_jobs.csv", index=False)
split.to_csv(f"../../data/splits/split_fake_jobs_seed{SEED}.csv", index=False)
print(split["split"].value_counts().to_dict())
print("Outlier-Rate je Split:", rows.groupby(split["split"])["fraudulent"].mean().round(4).to_dict())

## Encoding & Scaling – nur auf Train gefittet
- Median, Frequency-Encoding und StandardScaler sehen ausschließlich Trainingszeilen
- Im Train ungesehene Kategorien bekommen die Häufigkeit 0

In [ ]:
df["salary_avg"] = df["salary_avg"].fillna(df.loc[tr, "salary_avg"].median())

# frequency encoding fitted on train; categories unseen in train -> 0
for c in cat_cols:
    df[c] = df[c].map(df.loc[tr, c].value_counts(normalize=True)).fillna(0.0)

num_cols = ["salary_avg"] + binary + cat_cols
scaler = StandardScaler().fit(df.loc[tr, num_cols])
df[num_cols] = scaler.transform(df[num_cols])
print("Skaliert:", len(num_cols), "Spalten")

## Zusammenbauen & speichern

In [ ]:
out = df[["row_id"] + num_cols].reset_index(drop=True)
out["fraudulent"] = y.values
out.to_csv(f"../../data/preprocessed/cleaned_fake_jobs_seed{SEED}.csv", index=False)
print("Shape:", out.shape)

## Verifikation
- Gegenprobe zum Leakage: Scaler-Mittelwert auf Train ≈ 0, auf Test ≠ 0

In [ ]:
assert out.drop(columns=["row_id"]).isna().sum().sum() == 0
assert out["row_id"].is_unique
print("Mittelwert Train:", round(float(out.loc[tr.values, num_cols].to_numpy().mean()), 6))
print("Mittelwert Test :", round(float(out.loc[(split["split"] == "test").values, num_cols].to_numpy().mean()), 6))